In [6]:
import numpy, gsw

In [20]:
# ad-hoc implementation
def AOU_estimate(potential_temperature, salinity, density, oxygen):
    # function to compute AOU in umol/kg
    # based on Sarmiento & Gruber (Garcia and Gordon, 1992)
    # all inputs can be either scalar or 1D iterables of the same length
    # potential_temperature in degrees Celsius
    # salinity in situ in PSU
    # density in situ in kg/m3
    # oxygen in situ in umol/kg

    # convert to arrays; None -> nan
    pt = numpy.asarray(potential_temperature, dtype=float)
    sa = numpy.asarray(salinity, dtype=float)
    ro = numpy.asarray(density, dtype=float)
    o2 = numpy.asarray(oxygen, dtype=float)

    scalar = (pt.ndim == sa.ndim == ro.ndim == o2.ndim == 0)

    # force 1D so we only write the algebra once
    pt = numpy.atleast_1d(pt)
    sa = numpy.atleast_1d(sa)
    ro = numpy.atleast_1d(ro)
    o2 = numpy.atleast_1d(o2)

    # enforce equal shapes (no broadcasting surprises)
    if not (pt.shape == sa.shape == ro.shape == o2.shape):
        raise ValueError(f"Inputs must have the same shape; got {pt.shape}, {sa.shape}, {ro.shape}, {o2.shape}")

    # elementwise validity mask
    valid = numpy.isfinite(pt) & numpy.isfinite(sa) & numpy.isfinite(ro) & numpy.isfinite(o2)

    out = numpy.full(pt.shape, numpy.nan, dtype=float)

    # constants.  <-- KM: cm3/dm3 constants
    A0 = 2.00907
    A1 = 3.22014
    A2 = 4.05010
    A3 = 4.94457
    A4 = -0.256847
    A5 = 3.88767
    B0 = -6.24523e-3
    B1 = -7.37614e-3
    B2 = -1.03410e-2
    B3 = -8.17083e-3
    C0 = -4.88682e-7

    # compute only on valid positions
    ptv, sav, rov, o2v = pt[valid], sa[valid], ro[valid], o2[valid]

    Ts = numpy.log((298.15 - ptv) / (273.15 + ptv))
    l  = (A0
          + A1*Ts + A2*(Ts**2) + A3*(Ts**3) + A4*(Ts**4) + A5*(Ts**5)
          + sav*(B0 + B1*Ts + B2*(Ts**2) + B3*(Ts**3))
          + C0*(sav**2))

    O2_eq_mmol_per_m3 = (1000/22.3916) * numpy.exp(l)    # <-- KM: here we're converting using the molar volume of O2, but this I guess has some assumptions in it - STP?
    O2_eq_umol_per_kg = O2_eq_mmol_per_m3 * rov / 1000
    out[valid] = O2_eq_umol_per_kg - o2v

    if scalar:
        return float(out[0])
    return out

In [21]:
# gsw implementation
def AOU_estimate_gsw(SA, CT, p, lon, lat, oxygen):

    o2sol = gsw.O2sol(SA, CT, p, lon, lat)
    O2_eq_umol_per_kg = o2sol * gsw.rho(SA, CT, p) / 1000

    return O2_eq_umol_per_kg - oxygen

In [22]:
# example based off https://www.teos-10.org/pubs/gsw/html/gsw_O2sol.html:

SA = [34.7118, 34.8915, 35.0256, 34.8472, 34.7366, 34.7324]
CT = [28.8099, 28.4392, 22.7862, 10.2262, 6.8272, 4.3236]
p =  [10, 50, 125, 250, 600, 1000]
lat =  [4, 4, 4, 4, 4, 4]
long = [188, 188, 188, 188, 188, 188]
potential_temperature = gsw.pt_from_CT(SA, CT)
salinity = gsw.SP_from_SA(SA, p, long, lat)
density = gsw.rho(SA, CT, p)
oxygen = [0,0,0,0,0,0]

adhoc_AOU = AOU_estimate(potential_temperature, salinity, density, oxygen)
gsw_AOU = AOU_estimate_gsw(SA, CT, p, long, lat, oxygen)

In [23]:
print(adhoc_AOU)

[203.5708565  204.65385753 225.40507747 288.99983357 312.60233691
 332.28098609]


In [24]:
print(gsw_AOU)

[199.22315737 200.23354249 220.141552   281.47631455 304.32843056
 323.38669933]


In [3]:
# example from an argo profile
%pip install -i https://test.pypi.org/simple/ argovisHelpers==0.0.34a13

Looking in indexes: https://test.pypi.org/simple/
  Attempting uninstall: argovisHelpers
    Found existing installation: argovisHelpers 0.0.34a5
    Uninstalling argovisHelpers-0.0.34a5:
      Successfully uninstalled argovisHelpers-0.0.34a5
Note: you may need to restart the kernel to use updated packages.


In [1]:
from argovisHelpers import helpers as avh

In [77]:
qsp = {
    'id':'4902625_066',
    'data': 'all'
}
p = avh.Profile(avh.query('/argo',qsp)[0])

In [86]:
SA = gsw.SA_from_SP(p.getvar('salinity_sfile'), p.getvar('pressure'), p.longitude, p.latitude)
CT = gsw.CT_from_t(SA, p.getvar('temperature_sfile'), p.getvar('pressure'))
pressure =  p.getvar('pressure')
lat =  numpy.array([p.latitude for i in range(len(SA))])
long = numpy.array([p.longitude for i in range(len(SA))])
potential_temperature = gsw.pt_from_CT(SA, CT)
salinity = gsw.SP_from_SA(SA, pressure, long, lat)
density = gsw.rho(SA, CT, pressure)
oxygen = p.getvar('doxy')

mask = [x is not None for x in oxygen]

In [91]:
adhoc_AOU = AOU_estimate(potential_temperature[mask], salinity[mask], density[mask], oxygen[mask])
gsw_AOU = AOU_estimate_gsw(SA[mask], CT[mask], pressure[mask], long[mask], lat[mask], oxygen[mask])

In [90]:
adhoc_AOU

array([ 23.90140149,  23.92480198,  23.08045402,  23.07358378,
        23.14245095,  23.20363452,  23.43330514,  23.47100276,
        23.46644594,  23.5623651 ,  23.47517563,  23.62937336,
        23.64196805,  23.59834438,  23.73019402,  23.75454867,
        23.81222092,  23.88802645,  23.86403931,  24.0069963 ,
        24.06411945,  24.02750243,  24.14264582,  24.22171767,
        24.26002497,  24.28985138,  24.37387497,  24.48475072,
        24.31305027,  24.37618038,  24.29994854,  24.2017371 ,
        24.15081353,  23.83657236,  23.93058172,  23.69745239,
        23.45554739,  23.24000131,  23.24652051,  23.18701785,
        23.23725348,  22.92712439,  23.24434322,  23.57716178,
        23.56493101,  23.7469302 ,  23.92964361,  24.12239588,
        24.29142739,  24.6508106 ,  24.94433287,  25.48565415,
        25.71203198,  26.05616795,  26.35369548,  26.66626833,
        26.92032289,  27.19103484,  27.38859551,  27.47300413,
        27.65099756,  27.67138601,  27.86050588,  28.07

In [92]:
gsw_AOU

array([17.347763298857558, 17.36997974551224, 16.523264633212307,
       16.515210529414475, 16.582005239229403, 16.641951649074855,
       16.87101199825122, 16.90741408197607, 16.90061773260925,
       16.99501724704001, 16.90678179220444, 17.060612832944372,
       17.072656652271945, 17.027465075203423, 17.15833760055844,
       17.182043427747658, 17.23939096595859, 17.314904084379066,
       17.29047451971971, 17.43302887524399, 17.49009719764257,
       17.45342541719606, 17.568540300681804, 17.647682904371493,
       17.686460400072917, 17.716813303521207, 17.801340877489537,
       17.912013057111807, 17.738784319220997, 17.801306277999373,
       17.72523422795595, 17.6275134207379, 17.577207536225984,
       17.262754673406903, 17.356027024134278, 17.12079905585381,
       16.875509843794163, 16.650792042992236, 16.645398063707802,
       16.573737859196967, 16.604048172459073, 16.27211725082762,
       16.5815902076001, 16.910272666506728, 16.898371289223917,
       17.0805